# Bovine Pose Estimation — Interactive Reproducibility Audit & Verification

**File:**   
**Author:** Bert Klandermans (Data Scientist) — [ORCID: 0009-0009-2173-9422](https://orcid.org/0009-0009-2173-9422)  
**Project Lead:** Claudia Kamphuis — [ORCID: 0000-0001-5552-036X](https://orcid.org/0000-0001-5552-036X)  
**Affiliation:** Animal Breeding and Genomics, Wageningen Livestock Research (WUR)  
**Facility:** Dairy Campus, Leeuwarden, the Netherlands  
**Dataset DOI:** [10.5281/zenodo.23931569](https://doi.org/10.5281/zenodo.23931569)  
**Repository:** [dairycampus/dairycampus-cow-pose-estimation-klandermans](https://github.com/dairycampus/dairycampus-cow-pose-estimation-klandermans)  
**License:** CC-BY-4.0  

---
### Objective
This notebook provides an interactive testing and validation harness for the Dairy Campus bovine top-down pose estimation model (HRNet-W48). It enables researchers to:
1. Verify execution environment, GPU hardware, PyTorch, and OpenMMLab installations.
2. Inspect the COCO spatial annotations (300 train + 20 val images) and keypoint taxonomy.
3. Run inference on raw barn surveillance images with inline skeletal landmark visualization.
4. Execute quantitative provenance benchmarks against historical reference scores (PCK@0.05, PCK@0.10, PCK@0.20, mAP).
5. Render publication convergence curves and per-keypoint fingerprint alignment charts.

## 1. Environment & Hardware Verification
Verify Python, PyTorch, CUDA acceleration, and OpenMMLab framework availability.

In [ ]:
import os
import sys
import torch
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version:     {sys.version.split()[0]}")
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:         {torch.cuda.get_device_name(0)}")
    print(f"Device Capability:  {torch.cuda.get_device_capability(0)}")
    print(f"Device Memory:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

try:
    import mmengine
    import mmpose
    print(f"MMEngine Version:   {mmengine.__version__}")
    print(f"MMPose Version:     {mmpose.__version__}")
except ImportError as e:
    print(f"Notice: OpenMMLab dependencies not loaded: {e}")

## 2. Dataset Schema & Metadata Inspection
Inspect the COCO annotations for the 4 anatomical landmarks: , , , .

In [ ]:
import json

repo_root = os.path.dirname(os.path.abspath("")) if not os.path.exists("configs") else "."
val_ann_path = os.path.join(repo_root, "data/cow_barn_kpt_it2_corr_300_20/val/annotations/instances_default.json")

if os.path.exists(val_ann_path):
    with open(val_ann_path, "r", encoding="utf-8") as f:
        val_data = json.load(f)
    print(f"Dataset: {val_ann_path}")
    print(f"Validation Images:      {len(val_data['images'])}")
    print(f"Validation Annotations: {len(val_data['annotations'])}")
    cat = val_data['categories'][0]
    print(f"Category Name:          {cat['name']}")
    print(f"Keypoint Labels:        {cat['keypoints']}")
else:
    print("Notice: Full dataset not unpacked in local data/ folder. Using asset sample annotations.")

## 3. Top-Down Keypoint Visualization on Validation Barn Images
Visualize ground-truth and inferred skeletal landmarks on Dairy Campus surveillance frames.

In [ ]:
from PIL import Image, ImageDraw

sample_img = os.path.join(repo_root, "assets/sample_images/219_2023-03-02_09-00-03_21057.jpg")
if os.path.exists(sample_img):
    img = Image.open(sample_img)
    print(f"Loaded sample validation image: {sample_img}")
    print(f"Resolution: {img.size[0]} x {img.size[1]} px")
    
    plt.figure(figsize=(12, 7), dpi=150)
    plt.imshow(img)
    plt.title("Dairy Campus Barn View (Surveillance Cam Location 1)", fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.show()
else:
    print(f"Sample image not found: {sample_img}")

## 4. Quantitative Reproducibility Benchmark & Parity Audit
Compare the retrained HRNet-W48 model against historical WLR reference metrics logged in .

In [ ]:
csv_path = os.path.join(repo_root, "results/mmpose_benchmark_results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=== Official Provenance Benchmark Metrics ===")
    display(df)
else:
    print(f"Benchmark results file not found at: {csv_path}")

## 5. Visual Artifacts: Training Convergence & Keypoint Fingerprint
Render Figure 1 (Convergence Curve) and Figure 2 (PCK Fingerprint Alignment).

In [ ]:
from IPython.display import Image as IPImage, display as ip_display

fig1_path = os.path.join(repo_root, "assets/plots/figure1_training_convergence.png")
fig2_path = os.path.join(repo_root, "assets/plots/figure2_pck_fingerprint_match.png")
fig3_path = os.path.join(repo_root, "assets/plots/figure3_sample_pose_visualization.jpg")

if os.path.exists(fig2_path):
    print("--- Figure 2: PCK Metric Fingerprint Match ---")
    ip_display(IPImage(filename=fig2_path, width=700))
if os.path.exists(fig1_path):
    print("--- Figure 1: HPC 210-Epoch Training Convergence ---")
    ip_display(IPImage(filename=fig1_path, width=800))
if os.path.exists(fig3_path):
    print("--- Figure 3: Annotated Ground Truth Sample ---")
    ip_display(IPImage(filename=fig3_path, width=800))